In [1]:
# -*- coding: utf-8 -*-


# --- BEGIN TAGS PATCH (must run before sklearn is imported) ---
# --- BEGIN TAGS PATCH (must run before any SearchCV) ---
from sklearn.base import BaseEstimator
if not hasattr(BaseEstimator, "__sklearn_tags__"):
    BaseEstimator.__sklearn_tags__ = lambda self: {}

from scikeras.wrappers import KerasRegressor
if not hasattr(KerasRegressor, "__sklearn_tags__"):
    # 返回一个空 dict 或者根据需要设置标签
    KerasRegressor.__sklearn_tags__ = lambda self: {}
# --- END TAGS PATCH ---

import joblib
import os
# 一定要在 import tensorflow 之前设置
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0=ALL, 1=INFO, 2=WARNING, 3=ERROR
import os
# from openpyxl import load_workbook
from openpyxl import Workbook, load_workbook

EXCEL_PATH = 'model_results.xlsx'

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    mean_squared_log_error,
    explained_variance_score,
    max_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from skopt import BayesSearchCV
from tensorflow.keras import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense
from tensorflow.keras.layers import GRU
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
# from tensorflow.keras.wrappers.scikit_learn import KerasRegressor


from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from skopt import BayesSearchCV
import time
import argparse
from skopt.space import Real, Integer, Categorical
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 只保留 ERROR，不显示 INFO、WARNING

# 然后再 import
import warnings
warnings.filterwarnings('ignore')  # 全局忽略所有 warnings

import logging
tf.get_logger().setLevel(logging.ERROR)     # Keras/TF 自己的 logger
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# parser = argparse.ArgumentParser()
# parser.add_argument('--seed', type=int, default=42,
#                     help='random seed')
# args = parser.parse_args()
# random_seed = args.seed
random_seed = 42
np.random.seed(random_seed)
tf.random.set_seed(random_seed)

from sklearn.model_selection import GridSearchCV


print('-------------start!---------------------')

# === 1. read data
xls = pd.read_excel(
    '../ParleysRWIS_20211101-20230430.xlsx',
    sheet_name=['RwisAtmosphericData', 'RwisSurfaceData'],
    parse_dates=['SampleTime'],
    engine='openpyxl'
)
df1 = xls['RwisAtmosphericData']
df2 = xls['RwisSurfaceData']
df1['SampleTime'] = pd.to_datetime(df1['SampleTime'])
df2['SampleTime'] = pd.to_datetime(df2['SampleTime'])
df1['RwisControllerIntId'] = df1['RwisControllerIntId'].astype(str)
df2['RwisControllerIntId'] = df2['RwisControllerIntId'].astype(str)
df = pd.merge(df1, df2, on=['SampleTime','RwisControllerIntId'], how='inner', suffixes=('_atm','_surf'))

# === 2. delete useless feature
cols_to_drop = [
    'Pressure', 'SnowIntervalDepth','PrecipitationIntervalAmount','PrecipitationAmount',
    'RoadStatusText','iceCurCondIndex','iceAvgCondIndex','iceCurCondCode','iceAvgCondCode',
    'iceCurFricIndex','iceAvgFricIndex','iceCurFricCode','iceAvgFricCode','iceGrip','iceXValue',
    'iceYValue','iceYXRatio','iceLens','SurfaceSensorError','SurfaceSalinity','SurfaceFreezePoint','RoadStatus',
    'RoadStatus','RoadTemp','iceRoadTemp','RwisAtmoEquipmentIntId','WindSpeedGustTimestamp',
    'RwisSurfaceEquipmentIntId','WindDirection','WindSpeedAverage','WindSpeedGust','Visibility',
    'BatteryVoltage','dstInputVoltage','dstHardwareStatus','SurfaceStatusText','dscRoadStatusText','SubSurfaceTemp',
    'AirTemp_atm','DewPoint_atm','RelativeHumidity_atm','dscLevelOfGrip','dstRoadTemp','WinterRoadWeatherIndex',
    'StormIntensityIndex','PerformanceMetric','dscSurfStatus','WetBulbTemp'
]
df = df.drop(columns=cols_to_drop)

# === 3. remove outliers
filters = [
    (df['PrecipitationIntensity'] != 'fog'),
#     (df['WetBulbTemp'] >= -100),
    (df['SurfaceTemp'] < 200),
    (df['AirTemp_surf'] < 100),
    (df['DewPoint_surf'] < 80) & (df['DewPoint_surf'] > -20),
#     (df['dscSurfStatus'] < 10),
    (df['RainTotal'] <= 0.1),
    (df['SnowfallRate'] <= 3),
#     (df['WinterRoadWeatherIndex'] <= 1.6),
#     (df['StormIntensityIndex'] <= 2.2),
    (df['SurfaceWaterDepth'] <= 2),
    (df['SurfaceIceDepth'] <= 0.4),
    (df['SurfaceSnowDepth'] <= 2)
]
for f in filters:
    df = df[f]
df = df.fillna(df.mean(numeric_only=True))
df = df[(df['SurfaceGrip'] >= 0) & (df['SurfaceGrip'] <= 1)].reset_index(drop=True)

# === 4. choose time diff=10min data
df = df.sort_values(['RwisControllerIntId','SampleTime']).reset_index(drop=True)
df['time_diff'] = df.groupby('RwisControllerIntId')['SampleTime'].diff()
mask5 = ((df['time_diff']==pd.Timedelta(minutes=5)) & (df['SampleTime'].dt.minute%10==5))
df2 = df[~mask5].reset_index(drop=True)
df2 = df2.sort_values(['RwisControllerIntId','SampleTime']).reset_index(drop=True)
df2['time_diff'] = df2.groupby('RwisControllerIntId')['SampleTime'].diff()
df10 = df2[df2['time_diff']==pd.Timedelta(minutes=10)].reset_index(drop=True)

# === 5. One-hot, minmax
cat_cols = ['PrecipitationIntensity','PrecipitationType']
df_enc = pd.get_dummies(df10, columns=cat_cols)
label_col = 'SurfaceGrip'
feature_cols = [c for c in df_enc.columns if c not in ['SampleTime','RwisControllerIntId','time_diff',label_col]]
scaler = MinMaxScaler()
df_enc[feature_cols] = scaler.fit_transform(df_enc[feature_cols])

print('-------------finish data preprocessing!---------------------')

-------------start!---------------------
-------------finish data preprocessing!---------------------


In [3]:



# === 6. build dataset
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
def build_rnn_dataset(df, feature_cols, label_col='SurfaceGrip', window=25):
    X_seq, y_seq, groups = [], [], []
    for gid, grp in df.groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            hist = grp.iloc[i-window:i]
            tgt = grp.iloc[i]
            feat = hist[feature_cols].values
            hy = hist[[label_col]].values
            seq_hist = np.hstack([feat, hy])
            curr_feat = tgt[feature_cols].values.reshape(1,-1)
            curr_all = np.hstack([curr_feat, hy[-1:]])
            seq = np.vstack([seq_hist, curr_all])
            X_seq.append(seq); y_seq.append(tgt[label_col]); groups.append(gid)
    return np.array(X_seq), np.array(y_seq), np.array(groups)

def build_tabular_dataset(df, feature_cols, label_col='SurfaceGrip', window=25):
    X_tab, y_tab, groups = [], [], []
    for gid, grp in df.sort_values(['RwisControllerIntId','SampleTime']).groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            tgt = grp.iloc[i]
            X_tab.append(tgt[feature_cols].values)
            y_tab.append(tgt[label_col]); groups.append(gid)
    return np.array(X_tab), np.array(y_tab), np.array(groups)

X_seq, y_seq, groups_seq = build_rnn_dataset(df_enc, feature_cols, label_col)
X_tab, y_tab, groups_tab = build_tabular_dataset(df_enc, feature_cols, label_col)
assert np.array_equal(groups_seq, groups_tab)

# === 7. data balanced
def balance_y_distribution(X_seq, X_tab, y, bins=50, max_per_bin=1000, random_state=random_seed):
    np.random.seed(random_state)
    edges = np.linspace(np.min(y), np.max(y), bins+1)
    idx = np.digitize(y, bins=edges, right=True)
    sel = []
    for b in range(1, bins+1):
        ids = np.where(idx==b)[0]
        if len(ids)>max_per_bin:
            ids = np.random.choice(ids, max_per_bin, replace=False)
        sel.extend(ids)
    sel = np.array(sel)
    return X_seq[sel], X_tab[sel], y[sel]

# === 8. LOOCV ===
def evaluate(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    mae   = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true, y_pred)

    y_pred_pos = np.maximum(y_pred, 0)
    msle  = mean_squared_log_error(y_true, y_pred_pos)
    rmsle = np.sqrt(msle)

    smape = np.mean(
        np.abs(y_true - y_pred) /
        ((np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8)
    )

    if len(y_true) > 1:
        naive_mae = np.mean(np.abs(np.diff(y_true)))
        mase = mae / (naive_mae + 1e-8)
    else:
        mase = np.nan

    ev    = explained_variance_score(y_true, y_pred)
    mx    = max_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)

    return {
        'MSE':      mse,
        'RMSE':     rmse,
        'MAE':      mae,
        'MedAE':    medae,
        'MAPE':     mape,
        'sMAPE':    smape,
        'MSLE':     msle,
        'RMSLE':    rmsle,
        'MASE':     mase,
        'ExplVar':  ev,
        'MaxError': mx,
        'R2':       r2
    }
    
    



# --- 确保 init_model_excel 包含 dataset 列 ---
def init_model_excel(model_name):
    path = f'{model_name}_results.xlsx'
    if not os.path.exists(path):
        wb = Workbook()
        ws = wb.active
        ws.title = 'Metrics'
        # 表头：所有指标 + fold, station, dataset
        headers = list(evaluate(np.zeros(1), np.zeros(1)).keys()) + ['fold','station','dataset']
        ws.append(headers)
        wb.save(path)
    return path

def append_model_metrics(model_name, row_dict):
    path = f'{model_name}_results.xlsx'
    wb = load_workbook(path)
    ws = wb['Metrics']
    # 读表头
    header = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    # 按表头顺序排值
    row = [row_dict.get(col, "") for col in header]
    ws.append(row)
    wb.save(path)
    return path



# ——— LSTM ———
def train_lstm(X, y):
    def build_model(learning_rate=1e-3):
        m = Sequential([
            LSTM(64, input_shape=X.shape[1:]),
            Dense(1)
        ])
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)]
    )

    param_grid = {
        'epochs': [200, 300, 400]
    }
    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    # 搜索并 refit 最优模型
    grid.fit(X, y)
    best = grid.best_estimator_
    # 拿到训练过程的 History 对象
    hist = best.history_
    return best, hist




def loocv_run(X_seq, X_tab, y, groups):
    logo = LeaveOneGroupOut()

    tab_fns = [

    ]
    seq_fns = [
                ('LSTM_length25_seed42',    train_lstm)
        
    ]

    for fold, (tr, te) in enumerate(logo.split(X_tab, y, groups), start=1):
        if fold != 4:
            print('skip fold:',fold)
            continue
        station = groups[te][0]
        X_s_tr, X_t_tr, y_tr = X_seq[tr], X_tab[tr], y[tr]
        X_s_te, X_t_te, y_te = X_seq[te], X_tab[te], y[te]

        # 平衡
        X_s_tr, X_t_tr, y_tr = balance_y_distribution(X_s_tr, X_t_tr, y_tr)

        # 再从训练集中抽 20% 做验证
        X_t_tr_sub, X_t_val, y_tr_sub, y_val = train_test_split(
            X_t_tr, y_tr, test_size=0.2, random_state=random_seed
        )
        X_s_tr_sub, X_s_val, y_tr_sub_seq, y_val_seq = train_test_split(
            X_s_tr, y_tr, test_size=0.2, random_state=random_seed
        )

        print(f"\n=== Fold {fold}: leave station {station} ===")
        print('--- Tabular models ---')

        for name, fn in tab_fns:
            print(f'-----------{name}--------------')
            # 1) 训练（用 sub-train）
            mdl = fn(X_t_tr_sub, y_tr_sub)
            # 2) 验证集评估
            y_val_pred = mdl.predict(X_t_val)
            val_mets = evaluate(y_val, y_val_pred)
            val_mets.update({'model':name,'fold':fold,'station':station,'dataset':'val'})
            init_model_excel(name)
            append_model_metrics(name, val_mets)

            # 3) 测试集评估
            y_test_pred = mdl.predict(X_t_te)
            test_mets = evaluate(y_te, y_test_pred)
            test_mets.update({'model':name,'fold':fold,'station':station,'dataset':'test'})
            append_model_metrics(name, test_mets)

            # 4) 保存模型
            os.makedirs(f'models/{name}', exist_ok=True)
            joblib.dump(mdl, f'models/{name}/{name}_fold{fold}.joblib', compress=3)

            print(f"[{name}] val→{val_mets}, test→{test_mets}")

        print('--- Sequence models ---')

        for name, fn in seq_fns:
            print(f'-----------{name}--------------')
            # 1) 训练
            mdl, history = fn(X_s_tr_sub, y_tr_sub_seq)
            # 2) 验证集评估
            y_val_pred = mdl.predict(X_s_val).ravel()
            val_mets = evaluate(y_val_seq, y_val_pred)
            val_mets.update({'model':name,'fold':fold,'station':station,'dataset':'val'})
            init_model_excel(name)
            append_model_metrics(name, val_mets)

            # 3) 测试集评估
            y_test_pred = mdl.predict(X_s_te).ravel()
            test_mets = evaluate(y_te, y_test_pred)
            test_mets.update({'model':name,'fold':fold,'station':station,'dataset':'test'})
            append_model_metrics(name, test_mets)

            # 4) 保存 loss-vs-epoch 到同文件新 Sheet
#             df_hist = pd.DataFrame(history.history)
            df_hist = pd.DataFrame(history)
            path = f'{name}_results.xlsx'
            with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df_hist.to_excel(writer,
                                 sheet_name=f'History_Fold{fold}',
                                 index_label='epoch')

            # 5) 保存模型
            os.makedirs(f'models/{name}', exist_ok=True)
            mdl.model_.save(f'models/{name}/{name}_fold{fold}.h5')

            print(f"[{name}] val→{val_mets}, test→{test_mets}")
            
tf.autograph.set_verbosity(0)
os.makedirs('models', exist_ok=True)

print('-------------start model training!---------------------')
start_all = time.perf_counter()

# 不需要接返回值了，因为 loocv_run 已经负责写指标和存模型
loocv_run(X_seq, X_tab, y_tab, groups_seq)

end_all = time.perf_counter()
print(f"Total runtime: {end_all-start_all:.2f}s")
print("All folds done, metrics & models have been saved incrementally.")

-------------start model training!---------------------
skip fold: 1
skip fold: 2
skip fold: 3

=== Fold 4: leave station 29 ===
--- Tabular models ---
--- Sequence models ---
-----------LSTM_length25_seed42--------------
[LSTM_length25_seed42] val→{'MSE': 0.0011888513678412558, 'RMSE': 0.034479724010514586, 'MAE': 0.018816168102864166, 'MedAE': 0.009137570858001709, 'MAPE': 0.04288867904400787, 'sMAPE': 0.04247302455687404, 'MSLE': 0.000573356885961153, 'RMSLE': 0.02394487180924452, 'MASE': 0.12095161633682948, 'ExplVar': 0.954203376236049, 'MaxError': 0.3132183253765106, 'R2': 0.9537992716516285, 'model': 'LSTM_length25_seed42', 'fold': 4, 'station': '29', 'dataset': 'val'}, test→{'MSE': 0.0006619270142997357, 'RMSE': 0.025727942286543938, 'MAE': 0.01512592051141078, 'MedAE': 0.009044170379638672, 'MAPE': 0.022408461016598578, 'sMAPE': 0.02277566583339158, 'MSLE': 0.0002562051687110984, 'RMSLE': 0.0160064102381233, 'MASE': 2.4027148834986516, 'ExplVar': 0.9017659423004873, 'MaxError'